# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manahilrubabsatti/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face connection ready.")

Hugging Face connection ready.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

For the Refresh / Content Opportunity Scoring lane, I use page-level search and engagement signals from March 2026. The features describe performance that would be available at the decision moment.

I use impressions, clicks, average search position, pageviews, sessions, and engagement time. I also derive CTR and engagement rate from these observed fields.

I do not use client or content hash IDs as predictive features because they are identifiers rather than meaningful content signals.

In [4]:
# Build the feature vector from March 2026

query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,
    ga4_engaged_sessions,
    ga4_total_engagement_sec
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
    hive_partitioning=true
)
WHERE month = '2026-03'
"""

march = con.sql(query).df()

print("March rows:", len(march))
display(march.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 9841378


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,<NA>,<NA>


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
# Create engineered features

features = march.copy()

features["ctr"] = (
    features["gsc_clicks"] /
    features["gsc_impressions"].replace(0, pd.NA)
)

features["engagement_rate"] = (
    features["ga4_engaged_sessions"] /
    features["ga4_sessions"].replace(0, pd.NA)
)

features["avg_engagement_sec_per_session"] = (
    features["ga4_total_engagement_sec"] /
    features["ga4_sessions"].replace(0, pd.NA)
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ctr",
    "engagement_rate",
    "avg_engagement_sec_per_session"
]

feature_frame = features[[
    "report_date",
    "client_hash_id",
    "content_hash_id"
] + feature_cols].copy()

# Fill missing numeric feature values with 0
feature_frame[feature_cols] = (
    feature_frame[feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

Feature frame shape: (9841378, 11)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ctr,engagement_rate,avg_engagement_sec_per_session
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,0,0,0.000,0.0,0.0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0,0,0.000,0.0,0.0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,0,0,0.008,0.0,0.0
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,0,0,0.000,0.0,0.0
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,0,0,0.000,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

- `gsc_impressions`: search impressions observed before the decision moment; missing values are filled with 0.
- `gsc_clicks`: search clicks observed before the decision moment; missing values are filled with 0.
- `gsc_avg_position`: average search position; missing values are filled with 0, with the limitation that 0 is treated as missing/unknown.
- `ga4_pageviews`: observed pageviews before the decision moment; missing values are filled with 0.
- `ga4_sessions`: observed sessions before the decision moment; missing values are filled with 0.
- `ctr`: clicks divided by impressions; calculated only from observed search fields and filled with 0 when unavailable.
- `engagement_rate`: engaged sessions divided by sessions; calculated from observed fields and filled with 0 when unavailable.
- `avg_engagement_sec_per_session`: engagement seconds divided by sessions; calculated from observed fields and filled with 0 when unavailable.

All of these features are intended to describe information available before a content-refresh decision. No client or content hash is used as a predictive feature.

In [6]:
# Check missing values after feature preparation

print("Missing values in feature columns:")
display(feature_frame[feature_cols].isna().sum())

print("\nFeature columns:")
print(feature_cols)

Missing values in feature columns:


,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_pageviews,0
ga4_sessions,0
ctr,0
engagement_rate,0
avg_engagement_sec_per_session,0



Feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ctr', 'engagement_rate', 'avg_engagement_sec_per_session']


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I will not use a future outcome, a final label, or a field created directly from the outcome as a feature.

As a deliberate test, I create a leakage feature from the observed March data: `leak_is_high_click_page`. This uses clicks to create a label-like signal and is included only to demonstrate why outcome-derived information should not be used as a predictive feature.

The honest feature vector does not contain this column.

In [7]:
# Deliberate leakage experiment

leak_test = feature_frame.copy()

# Deliberately create a label-like column from an observed outcome.
# This is ONLY for demonstrating the leakage idea.
leak_test["leak_is_high_click_page"] = (
    leak_test["gsc_clicks"] >
    leak_test["gsc_clicks"].median()
).astype(int)

print("Leakage column created:")
display(
    leak_test[
        ["gsc_clicks", "leak_is_high_click_page"]
    ].head()
)

Leakage column created:


,gsc_clicks,leak_is_high_click_page
0,0,0
1,0,0
2,1,1
3,0,0
4,0,0


In [8]:
# Remove the deliberate leakage column

honest_features = leak_test.drop(
    columns=["leak_is_high_click_page"]
)

print("Leakage column present after removal:",
      "leak_is_high_click_page" in honest_features.columns)

print("\nFinal honest feature columns:")
print(
    [
        c for c in honest_features.columns
        if c not in ["report_date", "client_hash_id", "content_hash_id"]
    ]
)

Leakage column present after removal: False

Final honest feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ctr', 'engagement_rate', 'avg_engagement_sec_per_session']


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Fields I excluded

- `client_hash_id` — excluded because it is an identifier, not a meaningful performance feature.
- `content_hash_id` — excluded because it identifies the content item rather than describing its performance.
- Future-period performance fields — excluded because they would not be available at the decision moment and could leak the outcome.
- Any final outcome/label-derived field — excluded because it would make evaluation unrealistic.
- Private client names, domains, queries, or URLs — excluded to follow the dataset's privacy and education-use requirements.

The goal is to keep the feature vector based on information that could genuinely be available when deciding which pages deserve review.

In [9]:
# Final privacy/leakage check

excluded_or_identifier = [
    "client_hash_id",
    "content_hash_id"
]

predictive_features = [
    c for c in honest_features.columns
    if c not in ["report_date"] + excluded_or_identifier
]

print("Final predictive features:")
for col in predictive_features:
    print("-", col)

print("\nExcluded identifiers:")
for col in excluded_or_identifier:
    print("-", col)

Final predictive features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ctr
- engagement_rate
- avg_engagement_sec_per_session

Excluded identifiers:
- client_hash_id
- content_hash_id


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.